In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('imports OK')

---
## Problem 10: Off-Policy Evaluation & Safe Deployment

### Business Context
You've trained a new ranking model. Before rolling it out to 100% of users, you need to estimate its impact using historical logged data (no new A/B test yet) and design a safe staged deployment.

### Off-Policy Evaluation (OPE)
Estimate the value of a new policy π_e using data collected under a different (behavior) policy π_b.

**Three estimators:**

1. **Direct Method (DM)**: use a reward model to predict outcomes under π_e. Low variance but biased if model is wrong.
2. **Importance Sampling (IS)**: reweight logged rewards by π_e(a|s) / π_b(a|s). Unbiased but high variance.
3. **Doubly Robust (DR)**: combines DM and IS. Correct if EITHER the reward model OR IS weights are correct:
```
V_DR = E_D[Q(s,a)] + E_D[(π_e(a|s)/π_b(a|s)) * (r - Q(s,a))]
```

### Staged Deployment Gates
```
OPE check   -> V_DR(π_e) > V_DR(π_b) + δ with 95% CI > 0
Canary 1%   -> monitor for 48h (guardrail metrics: CTR, error rate)
Canary 10%  -> monitor for 1 week (A/B comparison)
Ramp 50%    -> monitor for 1 week
Full 100%   -> rollout complete
Rollback    -> if any guardrail breached, revert to previous version instantly
```

In [ ]:
def doubly_robust_estimator(rewards, actions, contexts,
                             behavior_probs, eval_policy_probs,
                             reward_model_preds, max_weight=20.0):
    """
    Doubly Robust Off-Policy Estimator.

    rewards:            observed rewards (n,)
    actions:            taken actions (n,)
    behavior_probs:     π_b(a|s) for taken actions (n,)
    eval_policy_probs:  π_e(a|s) for taken actions (n,)
    reward_model_preds: Q(s,a) predicted by reward model (n,)
    """
    n = len(rewards)

    # IS weights (clipped)
    is_weights = eval_policy_probs / np.clip(behavior_probs, 1e-6, None)
    is_weights = np.clip(is_weights, 0, max_weight)

    # Direct Method: mean of reward model predictions
    v_dm = reward_model_preds.mean()

    # DR correction: IS-weighted residuals
    residuals = rewards - reward_model_preds
    dr_correction = (is_weights * residuals).mean()

    v_dr = v_dm + dr_correction

    return {'V_DM': v_dm, 'V_DR': v_dr, 'IS_weight_mean': is_weights.mean()}

def bootstrap_confidence_interval(estimator_fn, data, n_bootstrap=1000, ci=0.95):
    """Bootstrap CI for any estimator function."""
    n = len(data['rewards'])
    estimates = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        boot_data = {k: v[idx] for k, v in data.items()}
        est = estimator_fn(**boot_data)
        estimates.append(est['V_DR'])

    alpha = (1 - ci) / 2
    return np.percentile(estimates, [alpha * 100, (1 - alpha) * 100])

# Simulation
np.random.seed(42)
n = 2000

# Logged data under behavior policy
behavior_probs  = np.random.uniform(0.05, 0.4, n)
eval_probs      = np.random.uniform(0.05, 0.4, n)   # new policy probabilities
rewards         = np.random.binomial(1, 0.1, n).astype(float)  # click/no-click
reward_preds    = np.random.uniform(0.05, 0.2, n)   # reward model predictions

# Behavior policy estimate
behavior_est = doubly_robust_estimator(
    rewards, None, None,
    behavior_probs, behavior_probs,   # IS weights = 1 when π_e = π_b
    reward_preds
)

# New policy estimate
eval_est = doubly_robust_estimator(
    rewards, None, None,
    behavior_probs, eval_probs,
    reward_preds
)

print('OPE Results:')
print(f'  Behavior policy V_DR: {behavior_est["V_DR"]:.4f}')
print(f'  Eval policy    V_DR:  {eval_est["V_DR"]:.4f}')
print(f'  Estimated lift: {(eval_est["V_DR"]/behavior_est["V_DR"] - 1)*100:+.1f}%')
print(f'  Avg IS weight:  {eval_est["IS_weight_mean"]:.2f}')

# Bootstrap CI
data = {'rewards': rewards, 'behavior_probs': behavior_probs,
        'eval_policy_probs': eval_probs, 'reward_model_preds': reward_preds}
# Wrapper to match bootstrap API
def est_wrapper(rewards, behavior_probs, eval_policy_probs, reward_model_preds):
    return doubly_robust_estimator(rewards, None, None, behavior_probs, eval_policy_probs, reward_model_preds)

ci_lo, ci_hi = bootstrap_confidence_interval(est_wrapper, data, n_bootstrap=500)
print(f'\n95% Bootstrap CI for V_DR: [{ci_lo:.4f}, {ci_hi:.4f}]')
print(f'Deploy recommendation: {"GO" if ci_lo > behavior_est["V_DR"] else "NO-GO (CI includes behavior policy)"}')

### L6 Interview Q&A

**Q: Why is Doubly Robust better than pure Importance Sampling?**  
A: IS is unbiased but has very high variance when π_e and π_b differ significantly — IS weights can be huge, making the estimate unstable. DR adds a control variate (the reward model prediction) that reduces variance while maintaining unbiasedness (the reward model bias is corrected by the IS term). DR is correct if either the IS weights are accurate OR the reward model is accurate — the 'double robustness' property.

**Q: Your bootstrap CI for the new policy is [+0.2%, +4.5%]. Do you deploy?**  
A: The lower bound is positive — the worst-case estimate still shows improvement. Proceed to a 1% canary with a 48-hour monitoring window on guardrail metrics (CTR, p99 latency, error rate). Don't interpret the CI as guaranteeing a real 0.2-4.5% lift — OPE estimates have their own model error. The CI gives directional confidence, not a guaranteed range.

**Q: When should you roll back during a staged deployment?**  
A: Pre-define rollback conditions before deployment: (1) any guardrail metric drops >X% vs. control (e.g. CTR -5%, conversion -3%), (2) error rate increases >0.1%, (3) p99 latency exceeds SLA, (4) unexpected prediction distribution shift (KL > 0.2 vs. baseline). Automated rollback is better than manual — define the conditions in code, not in a runbook.

### Common Pitfalls
- Not logging behavior policy probabilities π_b(a|s) at serving time — makes OPE impossible later
- Not clipping IS weights — one outlier can dominate the estimate
- Deploying directly to 100% without staged rollout — no recovery path if something breaks
- Using OPE as a hard gate rather than one signal among several